In [ ]:
%pip install git+https://github.com/Eickland/HumSpectra.git --upgrade

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import geopandas as gpd
from scipy.interpolate import griddata
import contextily as ctx

import HumSpectra.fluorescence as Hfl
import HumSpectra.ultraviolet as Huv
import HumSpectra.math_stat as Hst
import HumSpectra.geomodule as Hgeo

In [ ]:
wdir = Path.cwd().parent
wdir_data = Path.cwd().parent / "Исходные данные"

fluo_paths = Path.cwd().parent / "Исходные данные" / "2024" / "Флуоресценция"
uv_paths = Path.cwd().parent / "Исходные данные" / "2024" / "UV"

In [ ]:
fluo_spectra_list = [Hfl.read_fluo_3d(str(path)) for path in fluo_paths.rglob('*.csv')]
uv_spectra_list = [Huv.read_csv_uv(str(path),spectra_type="absorption") for path in uv_paths.rglob('*.csv')]

In [ ]:
toc_data = pd.read_excel(r'C:\Users\mnbv2\Desktop\lab\КНП\Исходные данные\2024\TOC.xlsx') # pyright: ignore[reportCallIssue]
map_data = pd.read_excel(r'C:\Users\mnbv2\Desktop\lab\КНП\Исходные данные\2024\Карты.xlsx')

In [ ]:
fl_data = pd.DataFrame()

fl_data["Sample"] = [fl_spectra.attrs['name']
    for fl_spectra in fluo_spectra_list]

fl_data["Class"] = [fl_spectra.attrs['class']
    for fl_spectra in fluo_spectra_list]

fl_data["Subclass"] = [fl_spectra.attrs['subclass']
    for fl_spectra in fluo_spectra_list]

fl_data["Asm350"] = [Hfl.asm_350(fl_spectra)
    for fl_spectra in fluo_spectra_list]

fl_data["Asm280"] = [Hfl.asm_280(fl_spectra)
    for fl_spectra in fluo_spectra_list]

fl_data["FIX"] = [Hfl.fluo_index(fl_spectra)
    for fl_spectra in fluo_spectra_list]

fl_data["HIX"] = [Hfl.humin_index(fl_spectra)
    for fl_spectra in fluo_spectra_list]

fl_data = fl_data[fl_data.Class == "ADOM"]

In [ ]:
uv_data = pd.DataFrame()

uv_data["Sample"] = [uv_spectra.attrs['name']
    for uv_spectra in uv_spectra_list]

uv_data["Class"] = [uv_spectra.attrs['class']
    for uv_spectra in uv_spectra_list]

uv_data["Subclass"] = [uv_spectra.attrs['subclass']
    for uv_spectra in uv_spectra_list]

uv_data["E2/E3"] = [Huv.e2_e3(uv_spectra)
    for uv_spectra in uv_spectra_list]

uv_data["epsilon"] = [Huv.epsilon(uv_spectra)
    for uv_spectra in uv_spectra_list]

uv_data = uv_data.merge(toc_data,on="Sample",how="inner")
uv_data = uv_data[uv_data.Class == "ADOM"]

uv_data["Suva"] = uv_data['epsilon']/uv_data['C_org']
uv_data.drop(columns=["epsilon"],inplace=True)

In [ ]:
MainData = uv_data.merge(fl_data,on=["Sample","Class","Subclass"],how="inner").merge(map_data,on=["Sample"],how="inner")

feature_columns = ["E2/E3","C_org","Suva","Asm350","Asm280"]

MainData.set_index(["Class","Subclass","Sample"],inplace=True)
MainData.head(3)

In [ ]:
vmin_list = []
vmax_list = []

storage_list = MainData.index.get_level_values(level=1).unique()
RawData = MainData.reset_index()
feature_columns.append("pH")

for col in feature_columns:
    # Вычисляем общие минимумы и максимумы для всего набора данных
    vmin_list.append(np.nanpercentile(MainData[col], 5))
    vmax_list.append(np.nanpercentile(MainData[col], 95))

for k, param in enumerate(feature_columns):
    fig, ax = plt.subplots(figsize=(12, 10))

    combined_gdf_list = []
    all_Z_values = []  # Собираем все значения интерполяции для корректного отображения цветовой шкалы

    for storage in storage_list:
        data = RawData[RawData.Subclass == storage]
        data = data.copy()

        data.dropna(inplace=True)
        data.drop_duplicates(subset=["lontitude", "lattitude"], inplace=True)

        # 1. Создаем GeoDataFrame в WGS84 (широта/долгота)
        points_ini = np.array(list(zip(data.lontitude, data.lattitude)))
        sorted_lon, sorted_lat = zip(*Hgeo.get_convex_hull_points(points_ini))

        for i, row in data.iterrows():
            if row.lontitude not in sorted_lon:
                data.drop(index=i, inplace=True)

        geometry = gpd.points_from_xy(sorted_lon, sorted_lat)
        gdf = gpd.GeoDataFrame(data, geometry=geometry, crs="EPSG:4326")

        # 2. Преобразуем в Web Mercator для contextily
        gdf_web = gdf.to_crs(epsg=3857)

        # 3. Создаем сетку для интерполяции в Web Mercator
        bounds = gdf_web.total_bounds
        x = np.linspace(bounds[0], bounds[2], 100)
        y = np.linspace(bounds[1], bounds[3], 100)
        X, Y = np.meshgrid(x, y)

        # 4. Интерполяция значений
        points = np.array([gdf_web.geometry.x, gdf_web.geometry.y]).T
        values = gdf_web[param].values
        Z = griddata(points, values, (X, Y), method='cubic')
        
        # Сохраняем значения интерполяции для настройки цветовой шкалы
        all_Z_values.append(Z)

        # 5. Визуализация - используем общие vmin/vmax для всех storage
        contour = ax.contourf(X, Y, Z, levels=20, alpha=0.7, cmap='RdYlBu_r', 
                             vmin=vmin_list[k], vmax=vmax_list[k])

        # Затем добавляем точки
        gdf_web.plot(ax=ax, color='black', markersize=50, edgecolor='white', linewidth=2)

        combined_gdf_list.append(gdf_web)

    # Объединяем все GeoDataFrame
    combined_gdf = gpd.GeoDataFrame(
        pd.concat(combined_gdf_list, ignore_index=True),
        crs=gdf_web.crs)  # type: ignore
    
    combined_gdf_web = combined_gdf.to_crs(epsg=3857)
    
    # Добавляем фоновую карту ПОВЕРХ всего
    ctx.add_basemap(ax, crs=combined_gdf_web.crs, alpha=0.8, source=ctx.providers.Esri.WorldImagery)  # type: ignore

    # КОРРЕКТНАЯ НАСТРОЙКА ЦВЕТОВОЙ ШКАЛЫ
    # Создаем нормализацию с общими значениями min/max
    norm = plt.Normalize(vmin=vmin_list[k], vmax=vmax_list[k]) # type: ignore
    
    # Создаем ScalarMappable для цветовой шкалы
    sm = plt.cm.ScalarMappable(cmap='RdYlBu_r', norm=norm)
    sm.set_array([])  # Пустой массив, так как данные уже отображены
    
    # Добавляем цветовую шкалу
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(f'{param}', rotation=270, labelpad=20,fontsize = 20)
    cbar.ax.tick_params(labelsize=10)
    
    # Устанавливаем корректные деления на цветовой шкале
    ticks = np.linspace(vmin_list[k], vmax_list[k], 5)  # 5 равномерно распределенных делений
    cbar.set_ticks(ticks)
    
    # Форматируем подписи делений
    cbar.set_ticklabels([f'{tick:.2f}' for tick in ticks],fontsize = 20)

    ax.set_title(f"2-6 карты {param}")
    plt.tight_layout()
    plt.show()

In [ ]:
def clustering(x):

    if x == "K2" or x == "K3":
        return "K2+K3"
    
    elif x == "K4" or x == "K7":
        return "K4+K7"
    
    else:
        return "K5+K6"

In [ ]:
PlotData = MainData.copy().reset_index()

PlotData = PlotData[PlotData.Suva < 0.1]
PlotData = PlotData[PlotData["E2/E3"] > 3]

target = "pH"

storage_names = ["K2","K3","K4","K5","K6","K7"]

PlotData["Карта"] = PlotData['Subclass'].apply(lambda x: ("K"+x[-1]))
PlotData["Карты"] = PlotData['Карта'].apply(lambda x: clustering(x))

for feature in feature_columns:

    sns.scatterplot(data=PlotData, x=target, y=feature,hue="Карты")
    plt.ylabel(r"$\frac{E_{2}}{E_{3}}$", fontsize=22)
    plt.xlabel(r"pH", fontsize=16)

    break